# Nhóm 08 – Notebook 02: Tìm kiếm siêu tham số (Grid, Random, TPE)

> Trước khi chạy: **Runtime → Change runtime type → T4 GPU**.

| Mục | Nội dung |
|---|---|
| Input | `common.py`, `config.yaml` (từ notebook 01, lấy về bằng `git clone`) |
| Output | `results/trials_grid.csv`, `results/trials_random.csv`, `results/trials_tpe.csv`, `results/trials_pareto.csv` |
| Purpose | Chạy ba phương pháp tìm kiếm dưới **cùng ngân sách B**, ghi lại **từng trial** để vẽ convergence và tái lập kết quả |

### Phân công trong notebook này

| Phần | Nội dung | Người làm |
|---|---|---|
| A–E | Thiết lập, hàm dùng chung, **Grid Search**, **Random Search** (10 seed) | SV1 (đã hoàn thành) |
| F | **TPE (Optuna)**, 10 seed | **SV2 hoàn thiện** (có sẵn khung và gợi ý TODO) |
| G | **Optuna đa mục tiêu** (Pareto accuracy – latency) | **SV2 hoàn thiện** |
| H | Tổng hợp nhanh và sao lưu | Cả nhóm |

### Nguyên tắc so sánh công bằng (giữ nguyên cho cả ba phương pháp)
- Cùng dữ liệu, cùng split, cùng hàm `train_eval`, cùng ngân sách **B = 32**.
- Seed huấn luyện mạng được **cố định** → f(x) là một hàm của x. Tính ngẫu nhiên chỉ đến từ **thuật toán tìm kiếm** (Random, TPE dùng 10 seed khác nhau).
- Chỉ dùng tập **validation** để chọn tham số. Không đụng đến test set ở notebook này.

In [1]:
# A. Lấy code dùng chung từ GitHub và cài thư viện
!pip -q install optuna pyyaml
![ -d /content/Nhom08_HPO_MLP_FashionMNIST ] || git clone https://github.com/nguyenhatiendat/Nhom08_HPO_MLP_FashionMNIST.git
%cd /content/Nhom08_HPO_MLP_FashionMNIST/NguyenHaTienDat_2474802010077
import os
os.makedirs('results', exist_ok=True)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 442.4/442.4 kB 13.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 268.7/268.7 kB 24.8 MB/s eta 0:00:00
Cloning into 'Nhom08_HPO_MLP_FashionMNIST'...
remote: Enumerating objects: 44, done.
remote: Counting objects: 100% (44/44), done.
remote: Compressing objects: 100% (39/39), done.
remote: Total 44 (delta 18), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (44/44), 24.38 KiB | 423.00 KiB/s, done.
Resolving deltas: 100% (18/18), done.
/content/Nhom08_HPO_MLP_FashionMNIST/NguyenHaTienDat_2474802010077


In [2]:
# B. Import, tải dữ liệu, đọc config
import itertools, time, numpy as np, pandas as pd
from common import *

data = load_data()
B, N_SEEDS = CFG['budget'], CFG['n_seeds']
SS, GRID = CFG['search_space'], CFG['grid']
print('Device:', DEVICE, '| Ngân sách B =', B, '| Số seed =', N_SEEDS)

100%|██████████| 26.4M/26.4M [00:01<00:00, 13.8MB/s]
100%|██████████| 29.5k/29.5k [00:00<00:00, 202kB/s]
100%|██████████| 4.42M/4.42M [00:01<00:00, 3.77MB/s]
100%|██████████| 5.15k/5.15k [00:00<00:00, 22.8MB/s]


Device: cuda | Ngân sách B = 32 | Số seed = 10


In [3]:
# C. Hàm dùng chung cho mọi phương pháp
def log_uniform(rng, lo, hi):
    return float(np.exp(rng.uniform(np.log(lo), np.log(hi))))

def sample_random(rng):
    # Lấy mẫu ngẫu nhiên trong search space (log-uniform cho lr, weight_decay, hidden)
    return dict(lr=log_uniform(rng, *SS['lr']),
                weight_decay=log_uniform(rng, *SS['weight_decay']),
                hidden=int(round(log_uniform(rng, *SS['hidden']))),
                dropout=float(rng.uniform(*SS['dropout'])),
                batch_size=int(rng.choice(SS['batch_size'])))

def grid_points():
    # Tích Descartes các giá trị lưới trong config: 2^5 = 32 điểm
    keys = list(GRID)
    return [dict(zip(keys, v)) for v in itertools.product(*[GRID[k] for k in keys])]

def evaluate(params, method, seed, trial):
    # MỘT lần đánh giá f(x): trả về 1 dòng log (siêu tham số + metric trên validation)
    m, _ = train_eval(params, data)
    return dict(method=method, seed=seed, trial=trial, **params, **m)

def add_best_so_far(df):
    # Cột best_val_acc = giá trị tốt nhất tính đến trial hiện tại (dùng vẽ convergence)
    df = df.copy()
    df['best_val_acc'] = df.groupby(['method', 'seed'])['val_acc'].cummax()
    return df

In [4]:
# D. Grid Search (tất định nên chỉ chạy 1 lần)
t0 = time.time()
pts = grid_points()
assert len(pts) == B, f'Lưới có {len(pts)} điểm nhưng ngân sách B = {B}'
grid_df = add_best_so_far(pd.DataFrame([evaluate(p, 'grid', 0, i) for i, p in enumerate(pts)]))
grid_df.to_csv('results/trials_grid.csv', index=False)
print(f'Grid xong sau {time.time() - t0:.0f}s | best val_acc = {grid_df.val_acc.max():.4f}')

Grid xong sau 32s | best val_acc = 0.8670


In [5]:
# E. Random Search (10 seed, mỗi seed B trial)
t0 = time.time()
rows = []
for seed in range(N_SEEDS):
    rng = np.random.RandomState(seed)
    for i in range(B):
        rows.append(evaluate(sample_random(rng), 'random', seed, i))
random_df = add_best_so_far(pd.DataFrame(rows))
random_df.to_csv('results/trials_random.csv', index=False)
best = random_df.groupby('seed').val_acc.max()
print(f'Random xong sau {time.time() - t0:.0f}s | best theo seed: mean = {best.mean():.4f}, std = {best.std():.4f}')

Random xong sau 279s | best theo seed: mean = 0.8580, std = 0.0041


## F. TPE (Optuna) — *SV2 hoàn thiện*

**Ý tưởng cần nắm để trả lời khi bảo vệ:** TPE dùng lịch sử các trial trước để chia các điểm đã thử thành nhóm "tốt" và "kém", rồi ưu tiên lấy mẫu ở vùng mà điểm tốt xuất hiện nhiều hơn (khác Random là lấy mẫu mù).

**Việc cần làm:** điền 3 chỗ `TODO` trong hàm `run_tpe`, sau đó đổi `RUN_TPE = True` và chạy.

Gợi ý API Optuna: `trial.suggest_float(name, low, high, log=True)`, `trial.suggest_int(name, low, high, log=True)`, `trial.suggest_categorical(name, choices)`, `optuna.create_study(direction=..., sampler=optuna.samplers.TPESampler(seed=...))`, `study.optimize(objective, n_trials=...)`.

Miền giá trị lấy từ `SS` (giống Random) để đảm bảo công bằng.

In [6]:
# F. TPE (Optuna) - SV2 hoàn thiện
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)
RUN_TPE = False   # đổi thành True khi đã làm xong các TODO

def run_tpe(seed):
    rows = []
    def objective(trial):
        # TODO 1: tạo dict params gồm lr, weight_decay, hidden, dropout, batch_size
        #         bằng trial.suggest_* (miền lấy từ SS; lr, weight_decay, hidden dùng log=True)
        params = None
        # TODO 2: gọi evaluate(params, 'tpe', seed, trial.number), thêm kết quả vào rows,
        #         rồi return giá trị val_acc để Optuna tối ưu
        return None
    # TODO 3: tạo study (maximize, sampler=TPESampler(seed=seed)) và chạy study.optimize(objective, n_trials=B)
    return rows

if RUN_TPE:
    t0 = time.time()
    tpe_rows = []
    for seed in range(N_SEEDS):
        tpe_rows += run_tpe(seed)
    tpe_df = add_best_so_far(pd.DataFrame(tpe_rows))
    tpe_df.to_csv('results/trials_tpe.csv', index=False)
    best = tpe_df.groupby('seed').val_acc.max()
    print(f'TPE xong sau {time.time() - t0:.0f}s | best theo seed: mean = {best.mean():.4f}, std = {best.std():.4f}')

## G. Optuna đa mục tiêu (Pareto accuracy – latency) — *SV2 hoàn thiện*

**Ý tưởng:** thay vì một mục tiêu, tối ưu **đồng thời** hai mục tiêu mâu thuẫn: tăng `val_acc` và giảm `latency_ms`. Kết quả không phải một nghiệm tốt nhất mà là một **tập nghiệm Pareto** (không nghiệm nào tốt hơn nghiệm khác ở cả hai mục tiêu).

**Việc cần làm:** hoàn thành `TODO` rồi đổi `RUN_PARETO = True`. Gợi ý: `optuna.create_study(directions=['maximize', 'minimize'], sampler=...)`, objective trả về **hai giá trị** `(val_acc, latency_ms)`, và `study.best_trials` cho tập Pareto. Chạy 1–3 seed là đủ (đây là phần mở rộng).

In [7]:
# G. Đa mục tiêu - SV2 hoàn thiện
RUN_PARETO = False   # đổi thành True khi đã làm xong TODO

def run_pareto(seed):
    rows = []
    def objective(trial):
        # TODO 4: giống TPE, tạo params bằng trial.suggest_*; gọi evaluate(params, 'pareto', seed, trial.number)
        #         rồi return (val_acc, latency_ms)
        return None
    # TODO 5: tạo study với directions=['maximize', 'minimize'], chạy n_trials=B
    return rows

if RUN_PARETO:
    pareto_df = pd.DataFrame(sum([run_pareto(s) for s in range(3)], []))
    pareto_df.to_csv('results/trials_pareto.csv', index=False)
    print(pareto_df[['val_acc', 'latency_ms', 'n_params']].describe().round(4))

In [8]:
# H. Tổng hợp nhanh (chi tiết và hình vẽ nằm ở notebook 03)
parts = [grid_df, random_df]
if RUN_TPE:
    parts.append(tpe_df)
all_df = pd.concat(parts)
final = all_df[all_df.trial == B - 1].groupby('method').best_val_acc.agg(['mean', 'std', 'min', 'max']).round(4)
print('best val_acc sau B trial (mean/std/min/max theo seed):')
print(final)
print('\nBaseline (notebook 01):')
print(pd.read_csv('results/baseline.csv').val_acc.agg(['mean', 'std']).round(4))

best val_acc sau B trial (mean/std/min/max theo seed):
         mean     std     min     max
method                               
grid    0.867     NaN  0.8670  0.8670
random  0.858  0.0041  0.8534  0.8644

Baseline (notebook 01):
mean    0.8575
std     0.0051
Name: val_acc, dtype: float64


In [10]:
# I. Sao lưu lên Google Drive (phiên Colab có thể bị ngắt)
from google.colab import drive
import shutil
drive.mount('/content/drive')
dst = '/content/drive/MyDrive/Nhom08_HPO'
os.makedirs(dst, exist_ok=True)
shutil.copytree('results', dst + '/results', dirs_exist_ok=True)
print('Đã lưu vào', dst)

Mounted at /content/drive
Đã lưu vào /content/drive/MyDrive/Nhom08_HPO
